In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install sentence-transformers umap-learn matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import entropy as scipy_entropy

model = SentenceTransformer("LaBSE")
print("Model loaded")

# ── Concept lexicon ───────────────────────────────────────────────────────────
# Each entry: concept_id → {language_code: word_or_phrase}
# Start with culturally-loaded abstract concepts. Extend this dict freely.
CONCEPTS = {
    "saudade":       {"pt":"saudade",        "en":"longing",         "es":"añoranza",     "de":"Sehnsucht",       "fr":"mélancolie",    "ja":"物悲しさ"},
    "hygge":         {"da":"hygge",           "en":"coziness",        "de":"Gemütlichkeit","fr":"convivialité",    "es":"acogedor",      "pt":"aconchego"},
    "schadenfreude": {"de":"Schadenfreude",   "en":"glee at misfortune","fr":"joie maligne","es":"regodeo",        "pt":"sadismo leve",  "ja":"他人の不幸を喜ぶ"},
    "wabi_sabi":     {"ja":"侘び寂び",         "en":"imperfect beauty", "de":"vergängliche Schönheit","es":"belleza imperfecta","fr":"beauté éphémère","pt":"beleza imperfeita"},
    "ubuntu":        {"zu":"ubuntu",           "en":"shared humanity", "es":"humanidad compartida","fr":"humanité commune","de":"Mitmenschlichkeit","pt":"humanidade coletiva"},
    "mamihlapinatapai":{"ona":"mamihlapinatapai","en":"mutual unspoken desire","es":"deseo mutuo no dicho","fr":"désir mutuel non exprimé","de":"gegenseitiges unausgesprochenes Verlangen","pt":"desejo mútuo não dito"},
    "age_otori":     {"ja":"あげおとり",        "en":"looking worse after a haircut","es":"peor después del corte","de":"schlechter nach dem Haarschnitt","fr":"pire après la coupe","pt":"pior após o corte"},
    "aware":         {"ja":"物の哀れ",          "en":"bittersweet impermanence","fr":"mélancolie douce","es":"melancolía dulce","de":"wehmütige Vergänglichkeit","pt":"impermanência agridoce"},
    "fernweh":       {"de":"Fernweh",          "en":"wanderlust",      "fr":"désir de voyage","es":"wanderlust",    "pt":"vontade de viajar","ja":"旅への渇望"},
    "meraki":        {"el":"μεράκι",           "en":"soul in your work","es":"alma en el trabajo","fr":"âme dans le travail","de":"Herzblut","pt":"alma no trabalho"},
    "joy":           {"en":"joy",              "es":"alegría",         "de":"Freude",       "fr":"joie",           "pt":"alegria",       "ja":"喜び"},  # control: should have LOW gap
    "table":         {"en":"table",            "es":"mesa",            "de":"Tisch",        "fr":"table",          "pt":"mesa",          "ja":"テーブル"},  # control: concrete noun
}

In [ ]:
# ── Embed everything ──────────────────────────────────────────────────────────
records = []
for concept_id, translations in CONCEPTS.items():
    for lang, word in translations.items():
        records.append({"concept": concept_id, "lang": lang, "word": word})

df = pd.DataFrame(records)
print(f"Embedding {len(df)} words across {df['lang'].nunique()} languages...")

embeddings = model.encode(df["word"].tolist(), normalize_embeddings=True, show_progress_bar=True)
df["embedding"] = list(embeddings)
print("Done.")

## Compute gap scores

In [ ]:
# ── Gap score algorithm ───────────────────────────────────────────────────────
# For each concept:
#   1. Compute the centroid across all language embeddings
#   2. Per-language distance to centroid = how far that language is from consensus
#   3. Gap score = std of those distances (high std = uneven coverage = real gap)
#   4. Also compute "isolation score" for each language = distance to centroid

results = []

for concept_id, group in df.groupby("concept"):
    embs   = np.stack(group["embedding"].values)
    langs  = group["lang"].values
    words  = group["word"].values
    
    centroid = embs.mean(axis=0)
    centroid /= np.linalg.norm(centroid)
    
    dists_to_centroid = cosine_distances(embs, centroid.reshape(1,-1)).flatten()
    
    # Pairwise distance matrix (how spread out are the translations?)
    pairwise = cosine_distances(embs)
    mean_pairwise = pairwise[np.triu_indices(len(embs), k=1)].mean()
    
    # Entropy of the distance distribution = how uneven the coverage is
    # (normalized distances as a distribution)
    dist_normed = dists_to_centroid / (dists_to_centroid.sum() + 1e-9)
    gap_entropy = scipy_entropy(dist_normed + 1e-9)
    
    results.append({
        "concept":          concept_id,
        "gap_score":        round(float(mean_pairwise), 4),
        "coverage_entropy": round(float(gap_entropy), 4),
        "n_languages":      len(langs),
        "per_lang_dist":    dict(zip(langs, dists_to_centroid.round(4))),
        "most_isolated_lang": langs[dists_to_centroid.argmax()],
        "most_isolated_word": words[dists_to_centroid.argmax()],
    })

results_df = pd.DataFrame(results).sort_values("gap_score", ascending=False)
print("\nConcept gap ranking (higher = harder to translate):\n")
print(results_df[["concept","gap_score","coverage_entropy","most_isolated_lang","most_isolated_word"]].to_string(index=False))

## Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Plot 1: Gap scores ranked ─────────────────────────────────────────────────
ax = axes[0]
colors = ["#E24B4A" if g > 0.3 else "#378ADD" if g > 0.15 else "#1D9E75"
          for g in results_df["gap_score"]]
bars = ax.barh(results_df["concept"], results_df["gap_score"], color=colors, height=0.6)
ax.axvline(x=0.15, color="gray", linestyle="--", alpha=0.5, label="low gap")
ax.axvline(x=0.30, color="gray", linestyle="-",  alpha=0.5, label="high gap")
ax.set_xlabel("Gap score (mean pairwise embedding distance)")
ax.set_title("Concept translatability — how spread out are the translations?")
ax.legend()
for bar, val in zip(bars, results_df["gap_score"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)

# ── Plot 2: Per-language isolation heatmap ───────────────────────────────────
ax = axes[1]
all_langs = sorted({l for r in results for l in r["per_lang_dist"].keys()})
heatmap_data = pd.DataFrame(
    {r["concept"]: {l: r["per_lang_dist"].get(l, np.nan) for l in all_langs}
     for r in results}
).T
sns.heatmap(heatmap_data, ax=ax, cmap="RdYlGn_r", annot=True, fmt=".2f",
            cbar_kws={"label": "Distance to centroid"}, linewidths=0.5)
ax.set_title("Per-language isolation\n(red = far from concept consensus)")
ax.set_xlabel("Language"); ax.set_ylabel("Concept")
plt.tight_layout()
plt.savefig("concept_gap_finder.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nInterpretation:")
print("Red cells = that language struggles to represent that concept.")
print("Whole red rows = genuinely untranslatable concept.")
print("Blue rows = concrete/universal concept (control check).")